Establish an environment

In [1]:
try:
  import google.colab

  IN_COLAB = True
except ImportError:
  IN_COLAB = False

if IN_COLAB:
  print("Running on Google Colab")
else:
  print("Running locally")

Running locally


In [2]:
!nvidia-smi

Sat Jan 31 11:12:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070        Off |   00000000:01:00.0  On |                  N/A |
|  0%   48C    P8             21W /  200W |    7095MiB /  12282MiB |     27%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Download data from roboflow API (API_KEY required) or from local machine

In [3]:
import os
from pathlib import Path

# Toggle between Roboflow API download or local pre-downloaded dataset
USE_ROBOFLOW_API = True

# Dataset: package_sorting
DATASET_NAME = "package_sorting"

ROBOFLOW_PROJECTS = {
  "package_sorting": {
    "workspace": "lukmik-e4qay",
    "project": "mailers-and-boxes-v10",
    "version": 6,
  },
}
FORMAT = "yolov11"

if USE_ROBOFLOW_API:
  if IN_COLAB:
    %pip install roboflow
    from google.colab import userdata

    ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
    DATASET_LOCATION = f"datasets/{FORMAT}"
  else:
    from dotenv import load_dotenv

    load_dotenv()
    ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY")
    DATASET_LOCATION = f"../datasets/{FORMAT}"

  from roboflow import Roboflow

  rf = Roboflow(api_key=ROBOFLOW_API_KEY)
  proj_config = ROBOFLOW_PROJECTS[DATASET_NAME]
  project = rf.workspace(proj_config["workspace"]).project(
    proj_config["project"]
  )
  version = project.version(proj_config["version"])
  dataset = version.download(FORMAT, location=DATASET_LOCATION)
  DATASET_PATH = Path(dataset.location)
else:
  if IN_COLAB:
    DATASET_PATH = Path(f"datasets/{DATASET_NAME}/yolo_format")
  else:
    DATASET_PATH = Path(f"../datasets/{DATASET_NAME}/yolo_format")

print(f"Dataset path: {DATASET_PATH.resolve()}")
print(f"Dataset exists: {DATASET_PATH.exists()}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to ../datasets/package_sorting/yolov11 in yolov11:: 100%|██████████| 1141/1141 [00:00<00:00, 19866.25it/s]

Dataset path: /home/lukmik/Desktop/university/masters2.1/wybrane_metody_uczenia_maszynowego/projekt/datasets/package_sorting/yolov11
Dataset exists: True


Install ultralytics

In [4]:
if IN_COLAB:
  %pip install ultralytics

Train model

In [ ]:
EPOCHS = 3

In [5]:
from pathlib import Path
import os
import glob

# Setup shared weights directory
if IN_COLAB:
  WEIGHTS_DIR = Path('weights')
else:
  WEIGHTS_DIR = (Path.cwd().parent / 'weights').resolve()
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# YOLOv11 base model
YOLO_MODEL = 'yolo11s.pt'
YOLO_WEIGHTS_PATH = WEIGHTS_DIR / YOLO_MODEL

# Download weights if they don't exist
if not YOLO_WEIGHTS_PATH.exists():
  print(f'Downloading {YOLO_MODEL}...')
  from ultralytics import YOLO

  # Temporarily change to weights directory for download
  original_dir = os.getcwd()
  os.chdir(WEIGHTS_DIR)

  try:
    model = YOLO(YOLO_MODEL)
    print(f'Downloaded to {YOLO_WEIGHTS_PATH}')
  finally:
    os.chdir(original_dir)

  # Clean up any stray .pt files in current directory
  for pt_file in glob.glob('*.pt'):
    if pt_file != YOLO_MODEL:
      target = WEIGHTS_DIR / pt_file
      if target.exists():
        os.remove(pt_file)
      else:
        os.rename(pt_file, target)
        print(f'Moved {pt_file} to weights/')
else:
  print(f'Using existing weights: {YOLO_WEIGHTS_PATH}')

OUTPUT_DIR = (Path.cwd().parent / "training_outputs" / "yolov11").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Change to weights directory before running yolo CLI to ensure downloads go there
%cd {WEIGHTS_DIR}
!yolo task=detect mode=train model={YOLO_WEIGHTS_PATH} data={DATASET_PATH}/data.yaml epochs={EPOCHS} imgsz=640 plots=True project={OUTPUT_DIR} name=runs/detect/train exist_ok=True
%cd -

New https://pypi.org/project/ultralytics/8.4.9 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.251 🚀 Python-3.13.1 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4070, 11864MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/lukmik/Desktop/university/masters2.1/wybrane_metody_uczenia_maszynowego/projekt/datasets/package_sorting/yolov11/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=tr